# Extended Sample Exploration
This notebook dynamically selects more books from the full library (`data/opium_books_fulltext`) to ensure we have at least 5 sample snippets for every opium-related keyword, while prioritizing keyword mentions across a **diverse set of books**, rather than pulling all 5 snippets from a single book.

In [2]:
import os
import re
import ast
import shutil
import pandas as pd
import spacy
from collections import defaultdict
from tqdm.auto import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

In [3]:
# Setup Paths
metadata_path = '../data/GP_1750_2000_opium_filtered.csv' # Changed this
fulltext_dir = '../data/opium_books_fulltext'
old_samples_dir = '../samples'
new_samples_dir = '../samples_extended'
output_csv = '../data/extended_sample_snippets.csv'

os.makedirs(new_samples_dir, exist_ok=True)
if os.path.exists(output_csv):
    os.remove(output_csv)

keywords = [
    'opium', 'poppy', 'anodyne', 'laudanum', 'morphine', 'codein', 'heroin',
    'narcotic', 'soporific', 'paregoric', 'dover’s powder', 'chandu',
    'nepenthe', 'dream-gum', 'dream-stick', 'black smoke'
]
target_snippets_per_keyword = 5
target_books_per_keyword = 5
window_size = 100

## 1. Load Metadata and Filter Out Old Samples

In [4]:
df_meta = pd.read_csv(metadata_path)
df_meta['Etext Number'] = df_meta['Etext Number'].astype(str)

existing_samples = set(os.listdir(old_samples_dir)) if os.path.exists(old_samples_dir) else set()

# Filter out books we already sampled
df_pool = df_meta[~df_meta['Etext Number'].isin(existing_samples)]
print(f"Total books available for new sampling: {len(df_pool)}")

Total books available for new sampling: 3812


## 2. Extract Snippets and Select Extended Sample Books
This cell implements a diverse selection algorithm. It ensures we don't pull all 5 occurrences from one single book if the keyword occurs in multiple other books.

In [5]:
# Pre-compute keywords dictionary for available books
book_info = {}
for idx, row in df_pool.iterrows():
    book_id = row['Etext Number']
    title = row['Title']
    kws = row['Opium Keywords']
    if pd.isna(kws): continue
    try:
        kw_list = ast.literal_eval(kws)
        book_info[book_id] = {'keywords': kw_list, 'title': title}
    except:
        pass

# Strategy: Sort books so that books with rarer keywords are checked first to guarantee coverage
global_kw_counts = {}
for b_info in book_info.values():
    for kw in b_info['keywords']:
        global_kw_counts[kw] = global_kw_counts.get(kw, 0) + 1

def book_rarity_score(book_id):
    kws = book_info[book_id]['keywords']
    if not kws: return float('inf')
    return min([global_kw_counts.get(kw, float('inf')) for kw in kws if kw in keywords] + [float('inf')])

sorted_book_ids = sorted(book_info.keys(), key=book_rarity_score)

snippets_by_kw = {kw: [] for kw in keywords}

for book_id in tqdm(sorted_book_ids, desc="Scanning Books"):
    kw_list = book_info.get(book_id, {}).get('keywords', [])
    
    # We need to process this book if it contains ANY keyword for which we haven't found snippets in enough DIFFERENT books
    needed = []
    for kw in kw_list:
        if kw not in keywords: continue
        distinct_books_for_kw = set(s['Book_ID'] for s in snippets_by_kw.get(kw, []))
        if len(distinct_books_for_kw) < target_books_per_keyword:
            needed.append(kw)
            
    if not needed:
        continue
        
    filepath = os.path.join(fulltext_dir, book_id)
    if not os.path.exists(filepath):
        if os.path.exists(filepath + '.txt'):
            filepath = filepath + '.txt'
        else:
            continue
            
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
        text_lower = text.lower()
        
        for keyword in needed:
            pattern = r'\b' + re.escape(keyword) + r'\b'
            
            # Find up to target_snippets_per_keyword matches in THIS book so we have fallbacks later
            matches = list(re.finditer(pattern, text_lower))[:target_snippets_per_keyword]
            for match in matches:
                idx = match.start()
                left_text = text[:idx]
                right_text = text[idx + len(keyword):]
                
                left_words = left_text.split()
                right_words = right_text.split()
                
                context_left = ' '.join(left_words[-window_size:])
                context_right = ' '.join(right_words[:window_size])
                
                snippets_by_kw[keyword].append({
                    'Book_ID': book_id,
                    'Title': book_info[book_id]['title'],
                    'Keyword': keyword,
                    'Snippet': f"{context_left} {text[idx:idx+len(keyword)]} {context_right}"
                })
                
    # Stop early if all keywords have reached target distinct books
    all_done = True
    for kw in keywords:
        if len(set(s['Book_ID'] for s in snippets_by_kw[kw])) < target_books_per_keyword and global_kw_counts.get(kw, 0) > 0:
            all_done = False
            break
    if all_done:
        print("Reached target distinct books for available keywords!")
        break

# Step 3: Diverse Selection Phase
final_results = []
selected_books = set()

print("\nDiverse Selection Phase:")
for kw in keywords:
    pool = snippets_by_kw[kw]
    if not pool: continue
    
    # Group by book
    by_book = defaultdict(list)
    for s in pool:
        by_book[s['Book_ID']].append(s)
        
    selected_for_kw = []
    book_ids = list(by_book.keys())
    
    # Round-Robin Selection to maximize diversity
    round_idx = 0
    while len(selected_for_kw) < target_snippets_per_keyword and book_ids:
        # iterate over a copy so we can remove exhausted books safely
        for bid in list(book_ids): 
            if len(by_book[bid]) > round_idx:
                selected_for_kw.append(by_book[bid][round_idx])
                if len(selected_for_kw) == target_snippets_per_keyword:
                    break
            else:
                book_ids.remove(bid)
        round_idx += 1
        
    final_results.extend(selected_for_kw)
    for s in selected_for_kw:
        selected_books.add(s['Book_ID'])
        # Ensure we copy the actual texts to samples_extended now that it's selected
        filepath = os.path.join(fulltext_dir, s['Book_ID'])
        if os.path.exists(filepath): shutil.copy2(filepath, os.path.join(new_samples_dir, os.path.basename(filepath)))
        elif os.path.exists(filepath + '.txt'): shutil.copy2(filepath + '.txt', os.path.join(new_samples_dir, os.path.basename(filepath + '.txt')))

df_results = pd.DataFrame(final_results)
df_results = df_results.sort_values(by=['Keyword', 'Book_ID']).reset_index(drop=True)

print(f"\nExtracted {len(df_results)} diverse snippets from {len(selected_books)} unique books.")

if len(df_results) > 0:
    df_results.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\nSnippets exported to {output_csv}")

Scanning Books:   0%|          | 0/3812 [00:00<?, ?it/s]


Diverse Selection Phase:

Extracted 65 diverse snippets from 38 unique books.

Snippets exported to ../data/extended_sample_snippets.csv


## 3. Manual Inspection Viewer
Use the slider to browse through the extracted snippets. The target keyword is highlighted.

In [ ]:
pd.set_option('display.max_colwidth', None)
def view_snippet(index):
    if len(df_results) == 0:
        print("No snippets found.")
        return
    row = df_results.iloc[index]
    
    # Highlight the keyword within the snippet using a case-insensitive match
    pattern = re.compile(re.escape(row['Keyword']), re.IGNORECASE)
    highlighted_text = pattern.sub(f"<span style='background-color: #ffeb3b; font-weight: bold; padding: 0 4px;'>\g<0></span>", row['Snippet'])
    
    html_out = f"""
    <div style='font-family: Georgia, serif; font-size: 16px; line-height: 1.6; max-width: 800px; padding: 20px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;'>
        <h4>Book ID: {row['Book_ID']} | Keyword: <span style='color: dimgrey;'>{row['Keyword'].upper()}</span></h4>
        <h5>{row['Title']}</h5>
        <hr>
        <p>{highlighted_text}</p>
    </div>
    """
    display(HTML(html_out))

if len(df_results) > 0:
    slider = widgets.IntSlider(min=0, max=len(df_results)-1, step=1, description='Snippet:', layout=widgets.Layout(width='800px'))
    widgets.interact(view_snippet, index=slider)

## 4. Entity & POS Exploration (Optional)
Analyze the named entities and adjectives co-occurring with opium keywords across this extended sample set.

In [ ]:
from collections import Counter
all_entities = []
all_adjectives = []

if len(df_results) > 0:
    for text in tqdm(df_results['Snippet'], desc="Processing NLP Contexts"):
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ in ['PERSON', 'ORG', 'GPE', 'LOC', 'FAC', 'PRODUCT']:
                all_entities.append((ent.label_, ent.text.strip()))
        for token in doc:
            if token.pos_ == 'ADJ' and not token.is_stop and token.is_alpha:
                all_adjectives.append(token.lemma_.lower())

    ent_counts = Counter(all_entities).most_common(20)
    adj_counts = Counter(all_adjectives).most_common(20)

    print("\n--- Top 20 Named Entities in Extended Contexts ---")
    for e, c in ent_counts:
        print(f"  {e[0]:<7}: {e[1]} ({c})")

    print("\n--- Top 20 Adjectives in Extended Contexts ---")
    for a, c in adj_counts:
        print(f"  {a:<15}: {c}")